In [1]:
import os, gc, warnings

warnings.filterwarnings("ignore")
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import glob

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from scipy.signal import butter, sosfilt
from sklearn.impute import SimpleImputer



In [2]:
PATH = "/kaggle/input/hms-harmful-brain-activity-classification/"
df = pd.read_csv(PATH + "train.csv")
TARGETS = df.columns[-6:]
print("Train shape:", df.shape)
print("Targets", list(TARGETS))
df.head()



Train shape: (96950, 15)
Targets ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']


,eeg_id,eeg_sub_id,eeg_label_offset_seconds,spectrogram_id,spectrogram_sub_id,spectrogram_label_offset_seconds,label_id,patient_id,expert_consensus,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,1628180742,0,0.0,353733,0,0.0,127492639,42516,Seizure,3,0,0,0,0,0
1,1628180742,1,6.0,353733,1,6.0,3887563113,42516,Seizure,3,0,0,0,0,0
2,1628180742,2,8.0,353733,2,8.0,1142670488,42516,Seizure,3,0,0,0,0,0
3,1628180742,3,18.0,353733,3,18.0,2718991173,42516,Seizure,3,0,0,0,0,0
4,1628180742,4,24.0,353733,4,24.0,3080632009,42516,Seizure,3,0,0,0,0,0


In [3]:
train = df.groupby("eeg_id")[
    ["spectrogram_id", "spectrogram_label_offset_seconds"]
].agg({"spectrogram_id": "first", "spectrogram_label_offset_seconds": "min"})
train.columns = ["spectrogram_id", "min"]

tmp = df.groupby("eeg_id")[["spectrogram_label_offset_seconds"]].agg("max")
train["max"] = tmp

tmp = df.groupby("eeg_id")[["patient_id"]].agg("first")
train["patient_id"] = tmp

tmp = df.groupby("eeg_id")[TARGETS].agg("sum")
for t in TARGETS:
    train[t] = tmp[t].values

y_data = train[TARGETS].values
y_data = y_data / y_data.sum(axis=1, keepdims=True)
train[TARGETS] = y_data

tmp = df.groupby("eeg_id")[["expert_consensus"]].agg("first")
train["target"] = tmp

train = train.reset_index()
print("Train non-overlapp eeg_id shape:", train.shape)
train.head(20)



Train non-overlapp eeg_id shape: (15396, 12)


,eeg_id,spectrogram_id,min,max,patient_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote,target
0,568657,789577333,0.0,16.0,20654,0.000000,0.000000,0.250000,0.000000,0.166667,0.583333,Other
1,582999,1552638400,0.0,38.0,20230,0.000000,0.857143,0.000000,0.071429,0.000000,0.071429,LPD
2,642382,14960202,1008.0,1032.0,5955,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,Other
3,1629671,2036345030,0.0,160.0,37481,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Seizure
4,1895581,128369999,1138.0,1138.0,47999,0.076923,0.000000,0.000000,0.000000,0.076923,0.846154,Other
5,2061593,320962633,1450.0,1450.0,23828,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,Other
6,2078097,2074135650,3342.0,3342.0,61174,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,Other
7,2366870,1232582129,0.0,30.0,23633,0.000000,0.333333,0.000000,0.000000,0.000000,0.666667,Other
8,2482631,978166025,1902.0,1944.0,20606,0.000000,0.000000,0.133333,0.066667,0.133333,0.666667,Other
9,2521897,673742515,0.0,4.0,62117,0.000000,0.000000,0.083333,0.083333,0.333333,0.500000,Other


In [4]:
ycol = ["seizure_vote", "lpd_vote", "gpd_vote", "lrda_vote", "grda_vote", "other_vote"]
cd = {
    "Seizure": "seizure_vote",
    "GPD": "gpd_vote",
    "LRDA": "lrda_vote",
    "Other": "other_vote",
    "GRDA": "grda_vote",
    "LPD": "lpd_vote",
}

eeg_id_col = train.iloc[:, 0]  # eeg_id
prob_cols = train.iloc[:, -7:-1]  # probability columns
label_col = train.iloc[:, -1]  # expert_consensus label

prob_cols = prob_cols.astype("float32")
prob_cols_normalized = prob_cols.div(prob_cols.sum(axis=1), axis=0)

normalized_data = pd.concat([eeg_id_col, prob_cols_normalized, label_col], axis=1)
normalized_data.head(20)



,eeg_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote,target
0,568657,0.000000,0.000000,0.250000,0.000000,0.166667,0.583333,Other
1,582999,0.000000,0.857143,0.000000,0.071429,0.000000,0.071429,LPD
2,642382,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,Other
3,1629671,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Seizure
4,1895581,0.076923,0.000000,0.000000,0.000000,0.076923,0.846154,Other
5,2061593,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,Other
6,2078097,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,Other
7,2366870,0.000000,0.333333,0.000000,0.000000,0.000000,0.666667,Other
8,2482631,0.000000,0.000000,0.133333,0.066667,0.133333,0.666667,Other
9,2521897,0.000000,0.000000,0.083333,0.083333,0.333333,0.500000,Other


In [5]:
normalized_data.to_csv("normalized_data.csv", index=False)



In [6]:
EEG_PATH = "/kaggle/input/hms-harmful-brain-activity-classification/train_eegs/"
train_path = "normalized_data.csv"  # use the CSV we just saved




In [7]:
class EEGDataset(Dataset):
    def __init__(self, csv_file, eeg_path):
        self.csv = pd.read_csv(csv_file)
        self.eeg_path = eeg_path
        self.sos = self.butter_bandpass_filter_init()
        self.FEATS = ["Fp1", "T3", "C3", "O1", "Fp2", "C4", "T4", "O2"]

    def butter_bandpass_filter_init(self):
        lowcut = 0.5
        highcut = 40.0
        fs = 200.0
        order = 5
        nyq = 0.5 * fs
        low = lowcut / nyq
        high = highcut / nyq
        sos = butter(order, [low, high], analog=False, btype="band", output="sos")
        return sos

    def butter_bandpass_filter(self, data):
        return sosfilt(self.sos, data)

    def __len__(self):
        return len(self.csv)

    def __getitem__(self, idx):
        eeg_id = self.csv.loc[idx, "eeg_id"]
        eeg_file_path = f"{self.eeg_path}{eeg_id}.parquet"
        eeg_data = pd.read_parquet(eeg_file_path)[self.FEATS].values

        if np.isnan(eeg_data).any():
            imputer = SimpleImputer(strategy="mean")
            eeg_data = imputer.fit_transform(eeg_data)

        eeg_data = self.butter_bandpass_filter(eeg_data)
        eeg_data = torch.tensor(eeg_data, dtype=torch.float32)

        mid_index = eeg_data.shape[0] // 2
        start_index = mid_index - 5000
        end_index = mid_index + 5000
        eeg_data = eeg_data[start_index:end_index]
        eeg_data = torch.transpose(eeg_data, 0, 1)  # shape: [channels, seq_len]

        labels = torch.tensor(
            self.csv.loc[idx, ycol].values.astype(np.float32), dtype=torch.float32
        )
        return eeg_data, labels


full_dataset = EEGDataset(train_path, EEG_PATH)

train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)



In [8]:
import torch.nn.functional as F


class TLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1, dropout=0.0):
        super(TLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers,
            dropout=dropout,
            bidirectional=True,
            batch_first=True,
        )

    def forward(self, x, timestamps=None):
        out, _ = self.lstm(x)
        return out, None


class EEGNet(nn.Module):
    def __init__(self, in_channels=8, num_classes=6):
        super(EEGNet, self).__init__()
        self.conv1 = nn.Conv1d(in_channels, 32, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.relu1 = nn.ReLU(inplace=True)
        self.pool1 = nn.MaxPool1d(2, 2)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.relu2 = nn.ReLU(inplace=True)
        self.pool2 = nn.MaxPool1d(2, 2)

        self.tlstm = TLSTM(input_size=64, hidden_size=128)

        self.attention = nn.Sequential(
            nn.Linear(128 * 2, 64), nn.Tanh(), nn.Linear(64, 1)
        )
        self.fc = nn.Linear(128 * 2, num_classes)

    def forward(self, x, timestamps=None):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        batch, channels, seq_len = x.size()
        x = x.permute(0, 2, 1)  # [batch, seq_len, channels]

        x, _ = self.tlstm(x, timestamps)  # x: [batch, seq_len, hidden*2]

        att_weights = F.softmax(self.attention(x), dim=1)  # [batch, seq_len, 1]
        x = torch.sum(att_weights * x, dim=1)  # [batch, hidden*2]

        x = self.fc(x)
        return x




In [9]:
input_channels = 8
num_classes = 6

model = EEGNet(in_channels=input_channels, num_classes=num_classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.KLDivLoss(reduction="batchmean")
optimizer = optim.Adam(
    model.parameters(), lr=0.0005
)  # reduced LR for steadier training

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.5)

best_val_loss = float("inf")
num_epochs = 15  # a few more epochs to let the model improve

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_dataloader:
        inputs = inputs.to(device)  # [batch, channels, seq_len]
        labels = labels.to(device)  # probability vectors

        optimizer.zero_grad()
        outputs = model(inputs, None)  # raw scores
        log_probs = F.log_softmax(outputs, dim=1)
        loss = criterion(log_probs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    train_loss = running_loss / len(train_dataloader)

    model.eval()
    val_running_loss = 0.0
    with torch.no_grad():
        for val_inputs, val_labels in val_dataloader:
            val_inputs = val_inputs.to(device)
            val_labels = val_labels.to(device)
            val_outputs = model(val_inputs, None)
            val_log_probs = F.log_softmax(val_outputs, dim=1)
            val_loss = criterion(val_log_probs, val_labels)
            val_running_loss += val_loss.item()
    val_loss = val_running_loss / len(val_dataloader)

    scheduler.step()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pt")

    print(
        f"Epoch {epoch+1}/{num_epochs} - TrainLoss: {train_loss:.6f} - ValLoss: {val_loss:.6f}"
    )

model.load_state_dict(torch.load("best_model.pt"))
print(f"Best validation KLDivLoss: {best_val_loss:.6f}")



Epoch 1/15 - TrainLoss: 1.239925 - ValLoss: 1.252506
Epoch 2/15 - TrainLoss: 1.217403 - ValLoss: 1.246563
Epoch 3/15 - TrainLoss: 1.213311 - ValLoss: 1.252415
Epoch 4/15 - TrainLoss: 1.200612 - ValLoss: 1.242876
Epoch 5/15 - TrainLoss: 1.172313 - ValLoss: 1.246718
Epoch 6/15 - TrainLoss: 1.127122 - ValLoss: 1.251955
Epoch 7/15 - TrainLoss: 1.095391 - ValLoss: 1.207457
Epoch 8/15 - TrainLoss: 1.085674 - ValLoss: 1.139939
Epoch 9/15 - TrainLoss: 1.067556 - ValLoss: 1.179584
Epoch 10/15 - TrainLoss: 1.055216 - ValLoss: 1.123072
Epoch 11/15 - TrainLoss: 1.046460 - ValLoss: 1.124510
Epoch 12/15 - TrainLoss: 1.035593 - ValLoss: 1.198568
Epoch 13/15 - TrainLoss: 1.030895 - ValLoss: 1.127750
Epoch 14/15 - TrainLoss: 1.027412 - ValLoss: 1.093083
Epoch 15/15 - TrainLoss: 1.019124 - ValLoss: 1.139808
Best validation KLDivLoss: 1.093083


In [10]:
test_path = "/kaggle/input/hms-harmful-brain-activity-classification/test.csv"
TEST_EEG_PATH = "/kaggle/input/hms-harmful-brain-activity-classification/test_eegs/"


class TestEEGDataset(Dataset):
    def __init__(self, csv_file, eeg_path):
        self.csv = pd.read_csv(csv_file)
        self.eeg_path = eeg_path
        self.sos = self.butter_bandpass_filter_init()
        self.FEATS = ["Fp1", "T3", "C3", "O1", "Fp2", "C4", "T4", "O2"]

    def __len__(self):
        return len(self.csv)

    def butter_bandpass_filter_init(self):
        lowcut = 0.5
        highcut = 40.0
        fs = 200.0
        order = 5
        nyq = 0.5 * fs
        low = lowcut / nyq
        high = highcut / nyq
        sos = butter(order, [low, high], analog=False, btype="band", output="sos")
        return sos

    def butter_bandpass_filter(self, data):
        return sosfilt(self.sos, data)

    def __getitem__(self, idx):
        eeg_id = self.csv.loc[idx, "eeg_id"]
        eeg_file_path = f"{self.eeg_path}{eeg_id}.parquet"
        eeg_data = pd.read_parquet(eeg_file_path)[self.FEATS].values
        eeg_data = self.butter_bandpass_filter(eeg_data)
        eeg_data = torch.tensor(eeg_data, dtype=torch.float32)

        mid = eeg_data.shape[0] // 2
        start = mid - 5000
        end = mid + 5000
        eeg_data = eeg_data[start:end]
        eeg_data = torch.transpose(eeg_data, 0, 1)
        return eeg_data


testdataset = TestEEGDataset(test_path, TEST_EEG_PATH)
test_dataloader = DataLoader(testdataset, batch_size=32, shuffle=False, num_workers=4)

model.eval()
predictions = []
with torch.no_grad():
    for inputs in test_dataloader:
        inputs = inputs.to(device)
        outputs = model(inputs, None)
        probs = torch.softmax(outputs, dim=1)
        predictions.append(probs.cpu().numpy())

predictions = np.concatenate(predictions, axis=0)
print("Predictions shape:", predictions.shape)



Predictions shape: (9850, 6)


In [11]:
# Ensure every row is a valid probability distribution
uniform_row = np.full(
    (predictions.shape[1],), 1.0 / predictions.shape[1], dtype=np.float32
)
predictions = np.where(np.isnan(predictions), uniform_row, predictions)
row_sums = predictions.sum(axis=1, keepdims=True)
predictions = predictions / np.clip(row_sums, a_min=1e-8, a_max=None)

columns = [
    "seizure_vote",
    "lpd_vote",
    "gpd_vote",
    "lrda_vote",
    "grda_vote",
    "other_vote",
]
results_df = pd.DataFrame(predictions, columns=columns)
print(results_df.head())



   seizure_vote  lpd_vote  gpd_vote  lrda_vote  grda_vote  other_vote
0      0.061142  0.108965  0.075029   0.103031   0.194962    0.456871
1      0.061142  0.108965  0.075029   0.103031   0.194962    0.456871
2      0.061142  0.108965  0.075029   0.103031   0.194962    0.456871
3      0.061142  0.108965  0.075029   0.103031   0.194962    0.456871
4      0.058094  0.069376  0.079382   0.033394   0.062334    0.697421


In [12]:
test_data = pd.read_csv(test_path)
sub = pd.DataFrame({"eeg_id": test_data["eeg_id"].values})

# Normalise again just in case and align column order with TARGETS
sub_probs = results_df.div(results_df.sum(axis=1), axis=0)
sub[TARGETS] = sub_probs[TARGETS]

# Write to the standard Kaggle working directory
submission_path = "submission.csv"
sub.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}, shape {sub.shape}")



Submission written to submission.csv, shape (9850, 7)


In [13]:
row_sums = sub.iloc[:, -6:].sum(axis=1)
print("Row sums -> min:", row_sums.min(), "max:", row_sums.max())

Row sums -> min: 0.9999998807907104 max: 1.0000001192092896
